# TinyDoc-VLM 768 Retrain (Kaggle)

Fully resumable full-model fine-tune on Kaggle GPU (free 30h/week).

Checkpoints + data live in private Hugging Face repos, so every Kaggle
session resumes from the last saved step. Add the **HF_TOKEN** secret:
left panel -> Add-ons -> Secrets.

Tune training with `--steps` etc. below. Runs single-GPU (T4): DDP/NCCL
native-SIGSEGVs on Kaggle T4 x2, so the notebook deliberately never sets
KAGGLE_DDP. `FRESH=1` (run_kaggle.sh) adds `--fresh` to start from the
clean init_768; without it the run resumes from the hub latest/ checkpoint.

In [ ]:
import subprocess, sys, os, time

REPO_URL = 'https://github.com/eulogik/TinyDoc-VLM'
REPO = '/kaggle/working/tinydoc-vlm'

STEPS = os.environ.get('STEPS', '9000')
BATCH = os.environ.get('BATCH', '2')
GRAD_ACCUM = os.environ.get('GRAD_ACCUM', '8')
LR = os.environ.get('LR', '0.00005')
WARMUP = os.environ.get('WARMUP', '0')
MAX_SEQ_LENGTH = os.environ.get('MAX_SEQ_LENGTH', '2048')
SCHEDULE_STEPS = os.environ.get('SCHEDULE_STEPS', '35000')

if not os.environ.get('HF_TOKEN'):
    try:
        from kaggle_secrets import UserSecretsClient
        _c = UserSecretsClient()
        for _ in range(5):
            try:
                os.environ['HF_TOKEN'] = _c.get_secret('HF_TOKEN')
                break
            except Exception:
                time.sleep(5)
    except Exception:
        pass

if not os.environ.get('HF_TOKEN'):
    raise RuntimeError('HF_TOKEN not set. Add it to Kaggle Secrets (Settings > Secrets).')

# Single-GPU by default: DDP/NCCL native-SIGSEGVs on Kaggle T4 x2 (rank 1, 20-30 min in; see kaggle_train.py).
# Set KAGGLE_DDP=1 only to experiment.

# Always fresh clone: Kaggle kernels keep /kaggle/working between re-runs
# in the same session, so a stale clone would miss self-healing/torch-pin
# fixes. depth-1 clone ~10s; kaggle_train.py re-clones anyway as backup.
if os.path.exists(REPO):
    subprocess.run(['rm', '-rf', REPO])
subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, REPO])

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'huggingface_hub'])

p = subprocess.run(
    [sys.executable, 'training/kaggle_train.py',
     '--steps', STEPS, '--batch-size', BATCH,
     '--grad-accum', GRAD_ACCUM, '--lr', LR, '--warmup', WARMUP,
     '--max-seq-length', MAX_SEQ_LENGTH, '--schedule-steps', SCHEDULE_STEPS],
    cwd=REPO,
)
sys.exit(p.returncode)